In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.ensemble import RandomForestClassifier

from sklearn.preprocessing import LabelEncoder

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

import matplotlib.pyplot as plt

import joblib

df = pd.read_csv(
    "../datasets/cleaned/startup_info.csv",
    low_memory=False
)

print(df.shape)

(50000, 84)


In [2]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

risk_cols = [

    "runway_months",
    "growth_score",
    "funding_strength_score",
    "retention_rate",
    "revenue_growth_rate",
    "customer_growth_rate"

]

for col in risk_cols:

    df[col+"_risk_norm"] = scaler.fit_transform(
        df[[col]]
    )

In [3]:
df["risk_score"] = (

      0.25 * df["runway_months_risk_norm"]

    + 0.20 * df["growth_score_risk_norm"]

    + 0.20 * df["funding_strength_score_risk_norm"]

    + 0.15 * df["retention_rate_risk_norm"]

    + 0.10 * df["revenue_growth_rate_risk_norm"]

    + 0.10 * df["customer_growth_rate_risk_norm"]

)

df["risk_score"] *= 100

In [4]:
df["risk_level"] = pd.cut(

    df["risk_score"],

    bins=[0,35,65,100],

    labels=[
        "High Risk",
        "Medium Risk",
        "Low Risk"
    ]

)

print(
    df["risk_level"]
    .value_counts()
)

risk_level
Medium Risk    39199
Low Risk        5689
High Risk       5112
Name: count, dtype: int64


In [5]:
le = LabelEncoder()

df["risk_level_encoded"] = (

    le.fit_transform(
        df["risk_level"]
    )

)

print(
    dict(
        zip(
            le.classes_,
            le.transform(le.classes_)
        )
    )
)

{'High Risk': 0, 'Low Risk': 1, 'Medium Risk': 2}


In [6]:
features = [

    "burn_rate",

    "runway_months",

    "growth_score",

    "funding_strength_score",

    "retention_rate",

    "revenue_growth_rate",

    "customer_growth_rate",

    "employee_growth_rate",

    "revenue",

    "customer_count"

]

X = df[features]

y = df["risk_level_encoded"]

In [7]:
X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.20,

    random_state=42,

    stratify=y
)

In [8]:
model = RandomForestClassifier(

    n_estimators=300,

    max_depth=12,

    random_state=42,

    n_jobs=-1

)

model.fit(
    X_train,
    y_train
)

print("Training Complete ✅")

Training Complete ✅


In [9]:
y_pred = model.predict(X_test)

In [10]:
acc = accuracy_score(
    y_test,
    y_pred
)

print(
    "Accuracy:",
    round(acc*100,2),
    "%"
)

Accuracy: 95.46 %


In [11]:
print(

    classification_report(

        y_test,

        y_pred,

        target_names=le.classes_

    )

)

              precision    recall  f1-score   support

   High Risk       0.97      0.81      0.89      1022
    Low Risk       0.97      0.81      0.89      1138
 Medium Risk       0.95      0.99      0.97      7840

    accuracy                           0.95     10000
   macro avg       0.97      0.87      0.91     10000
weighted avg       0.96      0.95      0.95     10000



In [12]:
importance = pd.DataFrame({

    "Feature": features,

    "Importance": model.feature_importances_

})

importance = importance.sort_values(

    by="Importance",

    ascending=False

)

print(importance)

                  Feature  Importance
1           runway_months    0.297688
2            growth_score    0.252918
3  funding_strength_score    0.126253
4          retention_rate    0.119584
5     revenue_growth_rate    0.060322
6    customer_growth_rate    0.057827
7    employee_growth_rate    0.036982
0               burn_rate    0.016838
8                 revenue    0.016013
9          customer_count    0.015573


In [14]:
joblib.dump(

    model,

    "../models/risk_model/risk_model.pkl"

)

joblib.dump(

    le,

    "../models/risk_model/label_encoder.pkl"

)

print("Risk Model Saved ✅")

Risk Model Saved ✅


In [15]:
import json

metadata = {

    "model_name": "Risk Analysis Model",

    "algorithm": "Random Forest",

    "accuracy": float(acc),

    "features": features

}

with open(

    "../models/risk_model/metadata.json",

    "w"

) as f:

    json.dump(
        metadata,
        f,
        indent=4
    )